In [1]:
from google.colab import files
uploaded = files.upload()

Saving matches.csv to matches.csv


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [4]:
df = pd.read_csv("matches.csv")
print(df.head())
print(df.info())
print(df.shape)
print(df.columns)


       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_decision  \
0        Kolkat

In [6]:
print(df.isnull().sum())

df['winner'] = df['winner'].fillna("No Result")

df['player_of_match'] = df['player_of_match'].fillna("Unknown")
df = df.drop_duplicates()

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       0
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                0
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64


In [8]:
print(df.describe(include='all'))
total_matches = df.shape[0]
print("Total Matches:", total_matches)
teams = pd.unique(df[['team1', 'team2']].values.ravel())
print("Total Teams:", len(teams))
print("Total Seasons:", df['season'].nunique())
most_wins = df['winner'].value_counts().idxmax()

print("Most Successful Team:", most_wins)


                  id season    city        date match_type player_of_match  \
count   1.095000e+03   1095    1044        1095       1095            1095   
unique           NaN     17      36         823          8             292   
top              NaN   2013  Mumbai  2008-04-26     League  AB de Villiers   
freq             NaN     76     173           2       1029              25   
mean    9.048283e+05    NaN     NaN         NaN        NaN             NaN   
std     3.677402e+05    NaN     NaN         NaN        NaN             NaN   
min     3.359820e+05    NaN     NaN         NaN        NaN             NaN   
25%     5.483315e+05    NaN     NaN         NaN        NaN             NaN   
50%     9.809610e+05    NaN     NaN         NaN        NaN             NaN   
75%     1.254062e+06    NaN     NaN         NaN        NaN             NaN   
max     1.426312e+06    NaN     NaN         NaN        NaN             NaN   

               venue                        team1           tea

In [10]:
season_matches = df['season'].value_counts().sort_index()

fig = px.bar(
    x=season_matches.index,
    y=season_matches.values,
    labels={'x':'Season', 'y':'Matches'},
    title='Matches Played Per Season'
)

fig.show()
team_wins = df['winner'].value_counts().head(10)

fig = px.bar(
    x=team_wins.index,
    y=team_wins.values,
    color=team_wins.values,
    title='Top Winning Teams'
)

fig.show()
toss_decision = df['toss_decision'].value_counts()

fig = px.pie(
    values=toss_decision.values,
    names=toss_decision.index,
    title='Toss Decision Distribution'
)

fig.show()



In [12]:
top_players = df['player_of_match'].value_counts().head(10)

fig = px.bar(
    x=top_players.index,
    y=top_players.values,
    color=top_players.values,
    title='Top Player of the Match Winners'
)

fig.show()
venue_matches = df['venue'].value_counts().head(10)

fig = px.bar(
    x=venue_matches.values,
    y=venue_matches.index,
    orientation='h',
    title='Top Venues by Match Count'
)

fig.show()


In [16]:
season = (input("Enter Season: "))

filtered_df = df[df['season'] == season]

print(filtered_df.head())
filtered_wins = filtered_df['winner'].value_counts()

fig = px.bar(
    x=filtered_wins.index,
    y=filtered_wins.values,
    title=f'Winning Teams in {season}'
)

fig.show()

Enter Season: 2007/08
       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_decis

In [17]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.0 MB/s eta 0:00:00


In [36]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

# Title
st.title("🏏 IPL Dashboard")

# Load Dataset
df = pd.read_csv("matches.csv")

# Dataset Preview
st.subheader("Dataset Preview")
st.write(df.head())

# Sidebar Filters
st.sidebar.header("Filters")

season = st.sidebar.selectbox(
    "Select Season",
    sorted(df['season'].dropna().unique())
)

# Filter Dataset
filtered_df = df[df['season'] == season]

# KPI Section
st.subheader("📊 Key Metrics")

col1, col2, col3 = st.columns(3)

col1.write("### Total Matches")
col1.write(filtered_df.shape[0])

col2.write("### Total Teams")
col2.write(filtered_df['winner'].nunique())

col3.write("### Top Winner")

if not filtered_df['winner'].mode().empty:
    col3.write(filtered_df['winner'].mode()[0])
else:
    col3.write("No Data")

# Winning Teams Chart
st.subheader("🏆 Winning Teams")

wins = filtered_df['winner'].value_counts()

fig = px.bar(
    x=wins.index,
    y=wins.values,
    labels={'x':'Teams', 'y':'Wins'},
    title='Winning Teams'
)

st.plotly_chart(fig)

# Toss Decision Chart
st.subheader("🪙 Toss Decision Analysis")

toss = filtered_df['toss_decision'].value_counts()

fig2 = px.pie(
    values=toss.values,
    names=toss.index,
    title='Toss Decisions'
)

st.plotly_chart(fig2)


Overwriting app.py


In [39]:
!pip install streamlit pyngrok

In [40]:
from pyngrok import ngrok

ngrok.set_auth_token("3ElGq66vJnqWb1jBrg6AKiCeNT1_24jNsgjK9JEG6SHXDMLXw")

In [ ]:
!streamlit run app.py &



2026-06-06 11:41:39.516 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.231.168.60:8501



In [42]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://contour-valid-console.ngrok-free.dev" -> "http://localhost:8501"
